# 02 — Voter registration 2026

**Owner:** B  
**Goal:** one row per municipality: registered voters by age group and gender, with snapshot_date

**Reads:** `data/raw/registration/`  
**Writes:** `INTERIM / "registration_2026.csv"`

Run the two setup cells below first, every time.

## Setup

In [ ]:
import sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Thato20-M/democratic-funnel.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/democratic-funnel")
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "src" / "config.py").exists())

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Repo:", REPO_DIR)

In [ ]:
from src import config
from src.config import RAW, INTERIM, PROCESSED, MODELS, FIGURES, SEED
from src.io_utils import read_any, clean_columns, inventory
from src.privacy import drop_personal

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings("ignore", category=FutureWarning)

config.make_dirs(); config.set_seed()
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid"); plt.rcParams["figure.dpi"] = 110
print(config.describe())

## Route check

If the 2026 entry on the results downloads page has registration figures, use it and skip the scrape.

In [ ]:
# inv = inventory(RAW / "registration"); inv

## Find the data endpoint

DevTools → Network → Fetch/XHR → select a municipality → Copy as cURL → curlconverter.com.

In [ ]:
import requests, time, json

# url = "..."            # from DevTools
# headers = {...}
# r = requests.get(url, headers=headers, timeout=30)
# r.json()

## Loop over all municipalities

One request per municipality, with a 1-second pause. Save the raw responses before parsing.

In [ ]:
# rows = []
# for prov in provinces:
#     for muni in munis[prov]:
#         rows.append(fetch(prov, muni)); time.sleep(1)

## Tidy into one row per municipality

Add registered_youth = 18_19 + 20_29, and stamp the snapshot date.

In [ ]:
from datetime import date
# reg["registered_youth"] = reg["registered_18_19"] + reg["registered_20_29"]
# reg["snapshot_date"] = date.today().isoformat()

## Check

National total should be near 29.2 million (21 Sept 2026 figure); it rises daily.

In [ ]:
# print(f'{reg["registered_total"].sum():,}')

## Save output

In [ ]:
# out.to_csv(INTERIM / "registration_2026.csv", index=False)
# print(out.shape); out.head()